# Mapping the AdTech Ecosystem with BigQuery Graphs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max-ostapenko/website-source/blob/main/src/posts/_drafts/ads_almanac/notebook.ipynb)

In [1]:
!pip install bigquery_magics==0.12.1 -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass  # Not running in Google Colab

import bigquery_magics
%load_ext bigquery_magics

bigquery_magics.context.project = 'httparchive'


## Build the Graph

The cells below create all base tables, node tables, edge tables, and the property graph
in the `scratchspace` dataset. Source: [HTTP Archive](https://httparchive.org/) monthly
crawl at `httparchive.crawl.pages`.

> **Development mode** — `rank <= 10000` limits the scan to roughly 1 % of the ~10 TiB
> source table. Remove the filter for a full monthly run.

In [3]:
%%bigquery
-- ============================================================
-- Step 1: Base tables
-- Source: httparchive.crawl.pages  |  date 2026-02-01  |  mobile  |  root pages
--
-- The //[ads] custom metric is stored in custom_metrics.other as JSON.
-- Path conventions inside that JSON object:
--   ads.txt data  →  $.ads.ads.*
--   sellers.json  →  $.ads.sellers.*
--   app-ads.txt   →  $.ads.app_ads.*  (not included in this analysis)
--
-- Crawl limitations relevant to this data:
--   • Google's sellers.json (~120 MB) is explicitly excluded: the JS metric
--     has a 5-second HTTP timeout + file-size limit. google.com therefore
--     appears in millions of ads.txt DIRECT entries but has no
--     RepresentsPublisher edges in this graph.
--   • Confidential sellers (is_confidential=1) are counted but their domains
--     are not stored — those publisher domains are invisible in this dataset.
--   • Domain deduplication only (JS Set): no seller_id is retained, so
--     cross-checks between ads.txt and sellers.json are domain-level only.
-- ============================================================

-- 1a. ads.txt relationships
CREATE OR REPLACE TABLE scratchspace.ads_txt_relationships AS
WITH pages AS (
  SELECT
    CASE page
      WHEN 'https://www.chunkbase.com/' THEN 'cafemedia.com'
      ELSE NET.REG_DOMAIN(page)
    END AS page,
    custom_metrics.other AS metrics
  FROM `httparchive.crawl.pages`
  WHERE date = '2026-02-01'
    AND client = 'mobile'
    AND is_root_page = TRUE
    AND rank <= 10000  -- remove for full crawl run
)
SELECT
  NET.REG_DOMAIN(REGEXP_EXTRACT(NORMALIZE_AND_CASEFOLD(domain),
    r'\b[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b')) AS platform_domain,
  page AS publisher_domain,
  'direct' AS relationship
FROM pages,
  UNNEST(JSON_VALUE_ARRAY(JSON_QUERY(metrics, '$.ads.ads.account_types'), '$.direct.domains')) AS domain
WHERE CAST(JSON_VALUE(metrics, '$.ads.ads.account_count') AS INT64) > 0

UNION ALL

SELECT
  NET.REG_DOMAIN(REGEXP_EXTRACT(NORMALIZE_AND_CASEFOLD(domain),
    r'\b[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b')) AS platform_domain,
  page AS publisher_domain,
  'reseller' AS relationship
FROM pages,
  UNNEST(JSON_VALUE_ARRAY(JSON_QUERY(metrics, '$.ads.ads.account_types'), '$.reseller.domains')) AS domain
WHERE CAST(JSON_VALUE(metrics, '$.ads.ads.account_count') AS INT64) > 0;

-- 1b. sellers.json relationships
-- $.both seller type means the partner acts as both publisher AND intermediary;
-- it is inserted into both groups so both RepresentsPublisher and SellsThrough
-- edges are populated.
CREATE OR REPLACE TABLE scratchspace.sellers_json_relationships AS
WITH pages AS (
  SELECT
    CASE page
      WHEN 'https://www.chunkbase.com/' THEN 'cafemedia.com'
      ELSE NET.REG_DOMAIN(page)
    END AS page,
    custom_metrics.other AS metrics
  FROM `httparchive.crawl.pages`
  WHERE date = '2026-02-01'
    AND client = 'mobile'
    AND is_root_page = TRUE
    AND rank <= 10000  -- remove for full crawl run
)
-- $.publisher → represents_publisher
SELECT page AS ssp_domain,
  NET.REG_DOMAIN(REGEXP_EXTRACT(NORMALIZE_AND_CASEFOLD(domain),
    r'\b[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b')) AS target_domain,
  'represents_publisher' AS seller_type
FROM pages,
  UNNEST(JSON_VALUE_ARRAY(JSON_QUERY(metrics, '$.ads.sellers.seller_types'), '$.publisher.domains')) AS domain
WHERE CAST(JSON_VALUE(metrics, '$.ads.sellers.seller_count') AS INT64) > 0

UNION ALL

-- $.both → represents_publisher direction
SELECT page AS ssp_domain,
  NET.REG_DOMAIN(REGEXP_EXTRACT(NORMALIZE_AND_CASEFOLD(domain),
    r'\b[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b')) AS target_domain,
  'represents_publisher' AS seller_type
FROM pages,
  UNNEST(JSON_VALUE_ARRAY(JSON_QUERY(metrics, '$.ads.sellers.seller_types'), '$.both.domains')) AS domain
WHERE CAST(JSON_VALUE(metrics, '$.ads.sellers.seller_count') AS INT64) > 0

UNION ALL

-- $.intermediary → intermediary
SELECT page AS ssp_domain,
  NET.REG_DOMAIN(REGEXP_EXTRACT(NORMALIZE_AND_CASEFOLD(domain),
    r'\b[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b')) AS target_domain,
  'intermediary' AS seller_type
FROM pages,
  UNNEST(JSON_VALUE_ARRAY(JSON_QUERY(metrics, '$.ads.sellers.seller_types'), '$.intermediary.domains')) AS domain
WHERE CAST(JSON_VALUE(metrics, '$.ads.sellers.seller_count') AS INT64) > 0

UNION ALL

-- $.both → intermediary direction
SELECT page AS ssp_domain,
  NET.REG_DOMAIN(REGEXP_EXTRACT(NORMALIZE_AND_CASEFOLD(domain),
    r'\b[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b')) AS target_domain,
  'intermediary' AS seller_type
FROM pages,
  UNNEST(JSON_VALUE_ARRAY(JSON_QUERY(metrics, '$.ads.sellers.seller_types'), '$.both.domains')) AS domain
WHERE CAST(JSON_VALUE(metrics, '$.ads.sellers.seller_count') AS INT64) > 0;


Query is running:   0%|          |

""


### Step 1c — Supplement with Google sellers.json

Google's `sellers.json` is ~120 MB and is excluded from the HTTP Archive crawl due to a
5-second HTTP fetch timeout in the JS custom metric. Since Google is the dominant ad
platform — appearing in millions of `ads.txt` DIRECT entries — we fix this gap by
downloading the file directly.

The cell downloads from Google's [canonical URL](https://storage.googleapis.com/adx-rtb-dictionaries/sellers.json),
loads raw domains into a staging table, then uses `NET.REG_DOMAIN()` inside BigQuery to
normalize them — keeping normalization identical to the rest of the pipeline — before
inserting into `scratchspace.sellers_json_relationships`.

In [4]:
import json
import requests
from google.cloud import bigquery

GOOGLE_SELLERS_URL = "https://storage.googleapis.com/adx-rtb-dictionaries/sellers.json"
PROJECT = "httparchive"
STAGING = f"{PROJECT}.scratchspace.google_sellers_staging"
TARGET = f"{PROJECT}.scratchspace.sellers_json_relationships"

print("Downloading Google sellers.json (~120 MB)…")
resp = requests.get(GOOGLE_SELLERS_URL, timeout=120)
resp.raise_for_status()
sellers = resp.json().get("sellers", [])

raw_rows = []
for s in sellers:
    if s.get("is_confidential", 0) == 1:
        continue
    domain = s.get("domain", "").strip().lower()
    if not domain:
        continue
    st = s.get("seller_type", "").upper()
    if st in ("PUBLISHER", "BOTH"):
        raw_rows.append({"raw_domain": domain, "seller_type": "represents_publisher"})
    if st in ("INTERMEDIARY", "BOTH"):
        raw_rows.append({"raw_domain": domain, "seller_type": "intermediary"})

print(f"Parsed {len(raw_rows):,} non-confidential entries from {len(sellers):,} total sellers")

client = bigquery.Client(project=PROJECT)

# Load raw domains to staging table (keeps Python-side logic minimal)
staging_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("raw_domain", "STRING"),
        bigquery.SchemaField("seller_type", "STRING"),
    ],
)
client.load_table_from_json(raw_rows, STAGING, job_config=staging_config).result()
print(f"Staged {len(raw_rows):,} rows — running NET.REG_DOMAIN normalization in BigQuery…")

# Normalize in BQ (consistent with the SQL pipeline) then append to main table
sql = (
    f"INSERT INTO `{TARGET}` (ssp_domain, target_domain, seller_type) "
    f"SELECT 'google.com', NET.REG_DOMAIN(raw_domain), seller_type "
    f"FROM `{STAGING}` WHERE NET.REG_DOMAIN(raw_domain) IS NOT NULL"
)
client.query(sql).result()

client.delete_table(STAGING, not_found_ok=True)
print(f"Done — Google sellers.json appended to {TARGET}")


Parsed 143,628 non-confidential entries from 991,615 total sellers
Staged 143,628 rows — running NET.REG_DOMAIN normalization in BigQuery…
Done — Google sellers.json appended to httparchive.scratchspace.sellers_json_relationships


In [5]:
%%bigquery
-- ============================================================
-- Step 2: Node tables
-- ============================================================

-- Publisher: web page domains that declare ads.txt or appear as publisher
--   entries in sellers.json. These are the inventory sources in the ecosystem.
CREATE OR REPLACE TABLE scratchspace.Publisher AS
SELECT DISTINCT domain
FROM (
  SELECT publisher_domain AS domain FROM scratchspace.ads_txt_relationships
  UNION DISTINCT
  -- SSPs declare these as publishers they represent; include as Publisher nodes.
  SELECT target_domain AS domain FROM scratchspace.sellers_json_relationships
  WHERE seller_type = 'represents_publisher'
)
WHERE domain IS NOT NULL;

ALTER TABLE scratchspace.Publisher ADD PRIMARY KEY (domain) NOT ENFORCED;

-- AdTechPlatform: DSPs, SSPs, and intermediary platforms.
--   A domain can appear in both Publisher and AdTechPlatform since many SSPs
--   also operate publisher properties alongside their platform business.
CREATE OR REPLACE TABLE scratchspace.AdTechPlatform AS
SELECT DISTINCT domain
FROM (
  SELECT platform_domain AS domain FROM scratchspace.ads_txt_relationships
  UNION DISTINCT
  SELECT ssp_domain AS domain FROM scratchspace.sellers_json_relationships
  UNION DISTINCT
  SELECT target_domain AS domain FROM scratchspace.sellers_json_relationships
  WHERE seller_type = 'intermediary'
)
WHERE domain IS NOT NULL;

ALTER TABLE scratchspace.AdTechPlatform ADD PRIMARY KEY (domain) NOT ENFORCED;

Query is running:   0%|          |

""


In [6]:
%%bigquery
-- ============================================================
-- Step 3: Edge tables
-- ============================================================

-- DirectBid: AdTechPlatform → Publisher
--   Source: ads.txt DIRECT. Publisher explicitly authorises the platform to sell
--   their inventory. Highest trust — typically commands higher CPMs.
CREATE OR REPLACE TABLE scratchspace.DirectBid AS
SELECT GENERATE_UUID() AS edge_id, platform_domain, publisher_domain
FROM scratchspace.ads_txt_relationships
WHERE relationship = 'direct'
  AND platform_domain IS NOT NULL
  AND publisher_domain IS NOT NULL;

ALTER TABLE scratchspace.DirectBid ADD PRIMARY KEY (edge_id) NOT ENFORCED;

-- ResellerBid: AdTechPlatform → Publisher
--   Source: ads.txt RESELLER. Publisher allows indirect reselling.
--   Reseller entries without a sellers.json counterpart are a compliance red flag.
CREATE OR REPLACE TABLE scratchspace.ResellerBid AS
SELECT GENERATE_UUID() AS edge_id, platform_domain, publisher_domain
FROM scratchspace.ads_txt_relationships
WHERE relationship = 'reseller'
  AND platform_domain IS NOT NULL
  AND publisher_domain IS NOT NULL;

ALTER TABLE scratchspace.ResellerBid ADD PRIMARY KEY (edge_id) NOT ENFORCED;

-- RepresentsPublisher: AdTechPlatform → Publisher
--   Source: sellers.json $.publisher and $.both. SSP claims to represent this publisher.
CREATE OR REPLACE TABLE scratchspace.RepresentsPublisher AS
SELECT GENERATE_UUID() AS edge_id, ssp_domain, target_domain AS publisher_domain
FROM scratchspace.sellers_json_relationships
WHERE seller_type = 'represents_publisher'
  AND ssp_domain IS NOT NULL
  AND target_domain IS NOT NULL;

ALTER TABLE scratchspace.RepresentsPublisher ADD PRIMARY KEY (edge_id) NOT ENFORCED;

-- SellsThrough: AdTechPlatform → AdTechPlatform
--   Source: sellers.json $.intermediary and $.both. Supply chain hop between SSPs.
--   These are the edges the original WITH RECURSIVE CTE was tracing depth-first.
CREATE OR REPLACE TABLE scratchspace.SellsThrough AS
SELECT GENERATE_UUID() AS edge_id, ssp_domain AS source_platform, target_domain AS dest_platform
FROM scratchspace.sellers_json_relationships
WHERE seller_type = 'intermediary'
  AND ssp_domain IS NOT NULL
  AND target_domain IS NOT NULL;

ALTER TABLE scratchspace.SellsThrough ADD PRIMARY KEY (edge_id) NOT ENFORCED;

Query is running:   0%|          |

""


In [7]:
%%bigquery
-- ============================================================
-- Step 4: Property Graph
-- ============================================================
CREATE OR REPLACE PROPERTY GRAPH scratchspace.AdsGraph
  NODE TABLES (
    scratchspace.Publisher
      KEY (domain)
      LABEL Publisher,
    scratchspace.AdTechPlatform
      KEY (domain)
      LABEL AdTechPlatform
  )
  EDGE TABLES (
    scratchspace.DirectBid
      KEY (edge_id)
      SOURCE KEY (platform_domain) REFERENCES AdTechPlatform (domain)
      DESTINATION KEY (publisher_domain) REFERENCES Publisher (domain)
      LABEL DirectBid,
    scratchspace.ResellerBid
      KEY (edge_id)
      SOURCE KEY (platform_domain) REFERENCES AdTechPlatform (domain)
      DESTINATION KEY (publisher_domain) REFERENCES Publisher (domain)
      LABEL ResellerBid,
    scratchspace.RepresentsPublisher
      KEY (edge_id)
      SOURCE KEY (ssp_domain) REFERENCES AdTechPlatform (domain)
      DESTINATION KEY (publisher_domain) REFERENCES Publisher (domain)
      LABEL RepresentsPublisher,
    scratchspace.SellsThrough
      KEY (edge_id)
      SOURCE KEY (source_platform) REFERENCES AdTechPlatform (domain)
      DESTINATION KEY (dest_platform) REFERENCES AdTechPlatform (domain)
      LABEL SellsThrough
  );

Query is running:   0%|          |

""


## Analysis

Each query uses BigQuery GQL (`GRAPH … MATCH`) to traverse the property graph, replacing
the original `WITH RECURSIVE` CTEs.

The graph has two node labels — **Publisher** and **AdTechPlatform** — and four edge
labels: **DirectBid**, **ResellerBid**, **RepresentsPublisher**, **SellsThrough**.

### Q1 — Direct publisher reach

Which demand platforms appear in the most publishers' `ads.txt` **DIRECT** entries?

`DIRECT` is the highest-trust record type: the publisher explicitly authorises the platform
to sell inventory without an intermediary. A large direct-reach score means broad, verified
relationships across the open web.

In [8]:
%%bigquery q1_direct
GRAPH scratchspace.AdsGraph
MATCH (platform:AdTechPlatform)-[:DirectBid]->(publisher:Publisher)
RETURN
  platform.domain AS platform,
  COUNT(DISTINCT publisher.domain) AS direct_reach
ORDER BY direct_reach DESC
LIMIT 50;

Query is running:   0%|          |

Downloading:   0%|          |

### Q2 — Reseller reach

Which platforms appear most often as **RESELLER** in publishers' `ads.txt`?

`RESELLER` entries have lower trust: the publisher allows the platform to resell inventory
through a third party. High reseller reach with low direct reach is a signal the platform
mainly operates as an aggregator rather than a direct partner.

In [9]:
%%bigquery q2_reseller
GRAPH scratchspace.AdsGraph
MATCH (platform:AdTechPlatform)-[:ResellerBid]->(publisher:Publisher)
RETURN
  platform.domain AS platform,
  COUNT(DISTINCT publisher.domain) AS reseller_reach
ORDER BY reseller_reach DESC
LIMIT 50;

Query is running:   0%|          |

Downloading:   0%|          |

### Q3 — Multi-hop supply chains

This is the GQL replacement for the original `WITH RECURSIVE` CTE. The `{1,3}` quantifier
traces each entry SSP through up to three `sellers.json` intermediary hops via
`SellsThrough` edges, preventing revisits automatically — no `STRPOS` cycle guard needed.

The `--graph` flag renders an interactive visualisation of the traversal paths.

In [10]:
%%bigquery --graph q3_supply_chain
GRAPH scratchspace.AdsGraph
MATCH
  (entry:AdTechPlatform)-[:ResellerBid]->(publisher:Publisher),
  (entry)-[:SellsThrough]->{1,3}(downstream:AdTechPlatform)
RETURN
  entry.domain AS entry_ssp,
  downstream.domain AS downstream_ssp,
  COUNT(DISTINCT publisher.domain) AS reachable_publishers
ORDER BY reachable_publishers DESC
LIMIT 100;

Query is running:   0%|          |

Downloading:   0%|          |

### Q4 — Cross-declaration consistency (ads.txt vs sellers.json)

An SSP listed as `RESELLER` in a publisher's `ads.txt` should ideally also list that
publisher in its own `sellers.json`. SSPs with high `reseller_reach` but low
`confirmed_in_sellers_json` are flagging a compliance gap — they claim reseller rights
not backed by their own declaration.

> **Note** — In the `rank <= 10000` dev run `confirmed_in_sellers_json` is near-zero
> because most SSP root domains (pubmatic.com, openx.com…) are not in the top-10k page
> crawl, so no `RepresentsPublisher` edges are generated. Run with the full crawl for
> meaningful divergence.

In [11]:
%%bigquery q4_consistency
GRAPH scratchspace.AdsGraph
MATCH (platform:AdTechPlatform)-[:ResellerBid]->(publisher:Publisher)
OPTIONAL MATCH (platform)-[confirms:RepresentsPublisher]->(publisher)
RETURN
  platform.domain,
  COUNT(DISTINCT publisher.domain) AS reseller_reach,
  COUNTIF(confirms IS NOT NULL) AS confirmed_in_sellers_json
ORDER BY reseller_reach DESC
LIMIT 50;

Query is running:   0%|          |

Downloading:   0%|          |